In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

df_censo = pd.read_csv("https://raw.githubusercontent.com/MarcilioFilh0/Data_Analysis_Education/refs/heads/Data_Table_Relationships/src/Datas/censo_filtrado.csv", sep=";")
df_evasao = pd.read_csv("https://raw.githubusercontent.com/MarcilioFilh0/Data_Analysis_Education/refs/heads/Data_Cleaning/src/Datas/Maiores_Taxas_Evasao_e_Reprovacao_2024.csv")

df_evasao['evasao_medio_total'] = pd.to_numeric(df_evasao['evasao_medio_total'].replace('Não informado', np.nan), errors='coerce')
df_evasao['evasao_fundamental_total'] = pd.to_numeric(df_evasao['evasao_fundamental_total'].replace('Não informado', np.nan), errors='coerce')

df_merged = pd.merge(df_censo, df_evasao, left_on='CO_ENTIDADE', right_on='codigo_escola', how='inner')

infra_map = {
    'IN_ENERGIA_REDE_PUBLICA': 'Energia Elétrica',
    'IN_AGUA_POTAVEL': 'Água Potável',
    'IN_INTERNET': 'Internet (Geral)',
    'IN_QUADRA_ESPORTES': 'Quadra de Esportes',
    'IN_LABORATORIO_CIENCIAS': 'Lab. de Ciências',
    'IN_REFEITORIO': 'Refeitório/Cantina',
    'IN_LABORATORIO_INFORMATICA': 'Lab. de Informática',
    'IN_INTERNET_ALUNOS': 'Internet para Alunos',
    'IN_ESGOTO_REDE_PUBLICA': 'Rede de Esgoto',
    'IN_SALA_LEITURA': 'Sala de Leitura/Biblioteca',
    'IN_ALIMENTACAO': 'Fornece Alimentação'
}

data_list = []

for col, name in infra_map.items():
    if col in df_merged.columns:
        grouped = df_merged.groupby(col)[['evasao_medio_total', 'evasao_fundamental_total']].mean().round(1)

        if 0.0 in grouped.index and 1.0 in grouped.index:
            medio_sem = grouped.loc[0.0, 'evasao_medio_total'] if not pd.isna(grouped.loc[0.0, 'evasao_medio_total']) else 0
            medio_com = grouped.loc[1.0, 'evasao_medio_total'] if not pd.isna(grouped.loc[1.0, 'evasao_medio_total']) else 0
            fund_sem = grouped.loc[0.0, 'evasao_fundamental_total'] if not pd.isna(grouped.loc[0.0, 'evasao_fundamental_total']) else 0
            fund_com = grouped.loc[1.0, 'evasao_fundamental_total'] if not pd.isna(grouped.loc[1.0, 'evasao_fundamental_total']) else 0

            data_list.append({
                'item': name,
                'medio_sem': float(medio_sem), 'medio_com': float(medio_com),
                'fund_sem': float(fund_sem), 'fund_com': float(fund_com),
                'impacto_medio': float(medio_sem - medio_com)
            })

df_plot = pd.DataFrame(data_list).sort_values(by='impacto_medio', ascending=True)

fig = go.Figure()

# ==========================================
# --- GERAÇÃO DOS GRÁFICOS (BARRAS) PARA O ENSINO MÉDIO ---
# ==========================================
fig.add_trace(go.Bar(
    y=df_plot['item'], x=df_plot['medio_sem'],
    name='Escolas SEM o item', orientation='h', marker_color='#ef553b',
    text=df_plot['medio_sem'], textposition='auto', visible=True
))
fig.add_trace(go.Bar(
    y=df_plot['item'], x=df_plot['medio_com'],
    name='Escolas COM o item', orientation='h', marker_color='#00cc96',
    text=df_plot['medio_com'], textposition='auto', visible=True
))

# ==========================================
# --- GERAÇÃO DOS GRÁFICOS (BARRAS) PARA O ENSINO FUNDAMENTAL ---
# ==========================================
fig.add_trace(go.Bar(
    y=df_plot['item'], x=df_plot['fund_sem'],
    name='Escolas SEM o item ', orientation='h', marker_color='#ef553b',
    text=df_plot['fund_sem'], textposition='auto', visible=False
))
fig.add_trace(go.Bar(
    y=df_plot['item'], x=df_plot['fund_com'],
    name='Escolas COM o item ', orientation='h', marker_color='#00cc96',
    text=df_plot['fund_com'], textposition='auto', visible=False
))

fig.update_layout(
    title='Impacto da Infraestrutura na Evasão Escolar - Ensino Médio',
    barmode='group',
    xaxis_title='Taxa Média de Evasão (%)',
    yaxis_title='Item de Infraestrutura',
    height=750,
    margin=dict(l=150, t=100),
    updatemenus=[
        dict(
            active=0,
            buttons=list([
                dict(label="Ensino Médio",
                     method="update",
                     args=[{"visible": [True, True, False, False]},
                           {"title": "Impacto da Infraestrutura na Evasão - Ensino Médio"}]),
                dict(label="Ensino Fundamental",
                     method="update",
                     args=[{"visible": [False, False, True, True]},
                           {"title": "Impacto da Infraestrutura na Evasão - Ensino Fundamental"}]),
            ]),
            x=0.5,
            xanchor="center",
            y=1.12,
            yanchor="top",
            direction="down",
            showactive=True,
            bgcolor="white",
            bordercolor="gray"
        )
    ]
)
fig.show()